In [1]:
!pip install firebase-admin python-dotenv

zsh:1: command not found: pip


In [2]:
!uv pip install firebase-admin python-dotenv

Using Python 3.12.3 environment at: /Users/sagar/dev/TheBrain/.venv
Resolved 35 packages in 724ms                                        
Prepared 22 packages in 2.92s                                            
Installed 23 packages in 25ms                               
 + cachecontrol==0.14.4
 + cryptography==46.0.7
 + firebase-admin==7.4.0
 + google-api-core==2.30.3
 + google-auth==2.49.2
 + google-cloud-core==2.5.1
 + google-cloud-firestore==2.27.0
 + google-cloud-storage==3.10.1
 + google-crc32c==1.8.0
 + google-resumable-media==2.8.2
 + googleapis-common-protos==1.74.0
 + grpcio==1.80.0
 + grpcio-status==1.80.0
 + h2==4.3.0
 + hpack==4.1.0
 + hyperframe==6.1.0
 + msgpack==1.1.2
 + proto-plus==1.27.2
 + protobuf==6.33.6
 + pyasn1==0.6.3
 + pyasn1-modules==0.4.2
 + pyjwt==2.12.1
 + python-dotenv==1.2.2


In [3]:
import firebase_admin
from firebase_admin import credentials, firestore
from dotenv import load_dotenv
import os
import json

# Load environment variables
load_dotenv()

# Path to your service account key
SERVICE_ACCOUNT_PATH = "../credentials/our-kitchen-app-5557e-firebase-adminsdk-fbsvc-7b99d2edb7.json"

# Initialize Firebase
if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_PATH)
    firebase_admin.initialize_app(cred)

db = firestore.client()
print("✅ Connected to Firestore successfully")

✅ Connected to Firestore successfully


In [4]:
# List all top-level collections in OurKitchen
collections = db.collections()
print("Collections in your Firestore database:")
for col in collections:
    print(f"  📁 {col.id}")

Collections in your Firestore database:
  📁 groceries
  📁 history
  📁 meals
  📁 ourlift_sessions
  📁 prefs


In [5]:
# Fetch all documents from the 'meals' collection
def get_all_meals():
    meals_ref = db.collection('meals')
    docs = meals_ref.stream()
    
    meals = []
    for doc in docs:
        meal = doc.to_dict()
        meal['id'] = doc.id  # attach the document ID
        meals.append(meal)
    
    print(f"Found {len(meals)} meals")
    return meals

meals = get_all_meals()

# Preview the first one to understand your data shape
if meals:
    print("\nSample meal document:")
    print(json.dumps(meals[0], indent=2, default=str))

Found 10 meals

Sample meal document:
{
  "cuisine": "Vietnamese",
  "name": "Bahn Mi Subs",
  "rating": 5,
  "fav": false,
  "ts": 1774991541622,
  "ingredients": [
    {
      "name": "chicken breast",
      "qty": "2 lbs"
    },
    {
      "name": "Vietnamese baguettes",
      "qty": "4 rolls"
    },
    {
      "name": "mayonnaise",
      "qty": "1/2 cup"
    },
    {
      "name": "cucumber",
      "qty": "2 medium"
    },
    {
      "name": "pickled daikon radish",
      "qty": "1 cup"
    },
    {
      "name": "pickled carrots",
      "qty": "1 cup"
    },
    {
      "name": "fresh cilantro",
      "qty": "1 bunch"
    },
    {
      "name": "jalape\u00f1o peppers",
      "qty": "2 whole"
    },
    {
      "name": "soy sauce",
      "qty": "3 tbsp"
    }
  ],
  "id": "1774991541622",
  "time": "",
  "proteins": [
    "Chicken"
  ],
  "note": ""
}


In [9]:
def get_weekly_planner():
    planner_ref = db.collection('history')  # adjust name if different
    docs = planner_ref.stream()
    
    weeks = []
    for doc in docs:
        week = doc.to_dict()
        week['id'] = doc.id
        weeks.append(week)
    
    print(f"Found {len(weeks)} weekly planner entries")
    return weeks

weeks = get_weekly_planner()
if weeks:
    print("\nSample weekly planner entry:")
    print(json.dumps(weeks[0], indent=2, default=str))

Found 2 weekly planner entries

Sample weekly planner entry:
{
  "2026-03-31_Dinner": {
    "dayKey": "2026-03-31",
    "id": "2026-03-31_Dinner",
    "mealId": "1774925587267",
    "slot": "Dinner"
  },
  "2026-04-02_Dinner": {
    "dayKey": "2026-04-02",
    "id": "2026-04-02_Dinner",
    "mealId": "1775093141237",
    "slot": "Dinner"
  },
  "id": "2026-03-30",
  "2026-04-03_Dinner": {
    "dayKey": "2026-04-03",
    "id": "2026-04-03_Dinner",
    "mealId": "1775058281194",
    "slot": "Dinner"
  },
  "2026-04-05_Dinner": {
    "dayKey": "2026-04-05",
    "id": "2026-04-05_Dinner",
    "mealId": "1775058281194",
    "slot": "Dinner"
  },
  "2026-04-05_Lunch": {
    "dayKey": "2026-04-05",
    "id": "2026-04-05_Lunch",
    "mealId": "1775093141237",
    "slot": "Lunch"
  },
  "2026-04-03_Lunch": {
    "dayKey": "2026-04-03",
    "id": "2026-04-03_Lunch",
    "mealId": "1774991681246",
    "slot": "Lunch"
  }
}


In [11]:
def get_preferences():
    # Preferences might be a single document — adjust path as needed
    prefs_ref = db.collection('prefs')
    docs = prefs_ref.stream()
    
    for doc in docs:
        print(f"Document ID: {doc.id}")
        print(json.dumps(doc.to_dict(), indent=2, default=str))

get_preferences()

Document ID: main
{
  "restrictions": [
    "no pork"
  ],
  "loves": []
}


In [12]:
def get_history_with_meals():
    # Build a lookup dictionary of meals by ID for fast access
    meals_ref = db.collection('meals').stream()
    meals_by_id = {}
    for doc in meals_ref:
        meal = doc.to_dict()
        meals_by_id[doc.id] = meal

    # Fetch all history documents (each = one week)
    history_ref = db.collection('history').stream()
    
    enriched_weeks = []
    for doc in history_ref:
        week = doc.to_dict()
        week_id = doc.id  # e.g. "2026-03-30"
        
        slots = []
        for key, value in week.items():
            if key == 'id':
                continue
            # Each value is a meal slot with a mealId
            if isinstance(value, dict) and 'mealId' in value:
                meal_id = value['mealId']
                meal = meals_by_id.get(meal_id, {})
                slots.append({
                    'date': value.get('dayKey'),
                    'slot': value.get('slot'),
                    'meal_name': meal.get('name', 'Unknown'),
                    'cuisine': meal.get('cuisine', ''),
                    'proteins': meal.get('proteins', []),
                    'ingredients': meal.get('ingredients', [])
                })
        
        # Sort slots by date
        slots.sort(key=lambda x: x['date'])
        
        enriched_weeks.append({
            'week_of': week_id,
            'meals': slots
        })
    
    enriched_weeks.sort(key=lambda x: x['week_of'])
    return enriched_weeks

history = get_history_with_meals()
print(json.dumps(history, indent=2, default=str))

[
  {
    "week_of": "2026-03-30",
    "meals": [
      {
        "date": "2026-03-31",
        "slot": "Dinner",
        "meal_name": "Unknown",
        "cuisine": "",
        "proteins": [],
        "ingredients": []
      },
      {
        "date": "2026-04-02",
        "slot": "Dinner",
        "meal_name": "Chicken Alfredo with Broccoli",
        "cuisine": "Italian",
        "proteins": [
          "Chicken"
        ],
        "ingredients": [
          {
            "name": "boneless chicken breast",
            "qty": "1 lb"
          },
          {
            "name": "fettuccine pasta",
            "qty": "12 oz"
          },
          {
            "name": "fresh broccoli florets",
            "qty": "3 cups"
          },
          {
            "name": "heavy cream",
            "qty": "1 cup"
          },
          {
            "name": "butter",
            "qty": "4 tbsp"
          },
          {
            "name": "freshly grated Parmesan cheese",
            "qty": "1